## 2. 序列模型

### 2.1 理论计算题

给定一个字符序列 "ababc"，假设采用一阶马尔可夫模型（即 $p(x_t|x_{t-1})$），使用拉普拉斯平滑（加 1 平滑）估计以下条件概率：
1. $p('a' | 'b')$
2. $p('c' | 'b')$

（词汇表为 {'a','b','c'}，计算时考虑所有可能转移，包括未出现的情况。）

### 2.1 解答

#### 步骤1：统计转移频次

序列 "ababc" 中的转移对有：
- $a \to b$（第1→2个字符）
- $b \to a$（第2→3个字符）
- $a \to b$（第3→4个字符）
- $b \to c$（第4→5个字符）

统计从字符 $b$ 出发的转移：
- $b \to a$：出现 1 次
- $b \to c$：出现 1 次
- $b \to b$：出现 0 次

$b$ 作为前一个字符的总次数：$\text{count}(b) = 1 + 1 + 0 = 2$

词汇表大小：$V = 3$

---

#### 步骤2：应用拉普拉斯平滑

拉普拉斯平滑（加1平滑）的条件概率公式为：
$$
p_{\text{Laplace}}(x|b) = \frac{\text{count}(b \to x) + 1}{\text{count}(b) + V}
$$

**问题1：$p('a' | 'b')$**
$$
p('a'|'b') = \frac{1 + 1}{2 + 3} = \frac{2}{5} = 0.4
$$

**问题2：$p('c' | 'b')$**
$$
p('c'|'b') = \frac{1 + 1}{2 + 3} = \frac{2}{5} = 0.4
$$

---

#### 验证

检查所有可能转移的概率之和是否为 1：
$$
p('a'|'b') + p('b'|'b') + p('c'|'b') = \frac{2}{5} + \frac{0 + 1}{5} + \frac{2}{5} = \frac{2}{5} + \frac{1}{5} + \frac{2}{5} = 1
$$

结果验证正确。

### 2.2 编程题

编写一个函数 `preprocess_text(text, n)`，完成以下步骤：
1. 将文本转换为小写，去除标点符号（保留字母和空格）。
2. 按空格分词。
3. 构建词汇表（按出现频率排序，分配整数 ID，从 0 开始）。
4. 用滑动窗口生成长度为 $n$ 的特征序列和对应的下一个词标签（用于自回归语言模型）。

返回词汇表字典和 (特征列表, 标签列表)。例如，输入 "The time machine" 和 $n=2$，应生成特征 `[['the','time'], ['time','machine']]` 和标签 `['machine', None]`（若无后续词则忽略）。

In [1]:
import re
from collections import Counter

def preprocess_text(text, n):
    """
    预处理文本，用于自回归语言模型。
    参数:
        text: 输入文本字符串
        n: n-gram 窗口大小
    返回:
        vocab: 词汇表字典 {词: id}
        (features, labels): 特征序列和标签序列
    """
    # 1. 转换为小写，去除标点符号（保留字母和空格）
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    
    # 2. 按空格分词
    tokens = text.split()
    
    # 3. 构建词汇表（按出现频率降序排序，分配整数ID从0开始）
    word_counts = Counter(tokens)
    vocab = {word: i for i, (word, _) in enumerate(word_counts.most_common())}
    
    # 4. 用滑动窗口生成特征序列和对应的下一个词标签
    features = []
    labels = []
    for i in range(len(tokens) - n):
        features.append(tokens[i:i+n])
        labels.append(tokens[i+n])
    # 若无后续词则忽略（不添加 None）
    
    return vocab, (features, labels)

# 测试示例
if __name__ == "__main__":
    # 示例1
    text1 = "The time machine"
    vocab1, (feat1, lab1) = preprocess_text(text1, n=2)
    print("=== 示例1 ===")
    print("词汇表:", vocab1)
    print("特征序列:", feat1)
    print("标签序列:", lab1)
    
    # 示例2：更长的文本
    text2 = "The quick brown fox jumps over the lazy dog"
    vocab2, (feat2, lab2) = preprocess_text(text2, n=3)
    print("\n=== 示例2 ===")
    print("词汇表:", vocab2)
    print("特征序列:", feat2)
    print("标签序列:", lab2)
    
    # 示例3：含标点的文本
    text3 = "Hello, World! This is a test."
    vocab3, (feat3, lab3) = preprocess_text(text3, n=2)
    print("\n=== 示例3（含标点） ===")
    print("词汇表:", vocab3)
    print("特征序列:", feat3)
    print("标签序列:", lab3)

=== 示例1 ===
词汇表: {'the': 0, 'time': 1, 'machine': 2}
特征序列: [['the', 'time']]
标签序列: ['machine']

=== 示例2 ===
词汇表: {'the': 0, 'quick': 1, 'brown': 2, 'fox': 3, 'jumps': 4, 'over': 5, 'lazy': 6, 'dog': 7}
特征序列: [['the', 'quick', 'brown'], ['quick', 'brown', 'fox'], ['brown', 'fox', 'jumps'], ['fox', 'jumps', 'over'], ['jumps', 'over', 'the'], ['over', 'the', 'lazy']]
标签序列: ['fox', 'jumps', 'over', 'the', 'lazy', 'dog']

=== 示例3（含标点） ===
词汇表: {'hello': 0, 'world': 1, 'this': 2, 'is': 3, 'a': 4, 'test': 5}
特征序列: [['hello', 'world'], ['world', 'this'], ['this', 'is'], ['is', 'a']]
标签序列: ['this', 'is', 'a', 'test']


## 3. 循环神经网络

### 3.1 理论计算题

考虑一个线性 RNN（无偏置），定义为 $h_t = W_{hh}h_{t-1} + W_{hx}x_t$，输出 $o_t = W_{oh}h_t$。假设损失函数为平方损失 $L = \frac{1}{2}\sum_{t=1}^{T}(o_t - y_t)^2$。推导损失对权重 $W_{hh}$ 的梯度表达式（通过时间反向传播，展开到所有时间步），并说明梯度消失或爆炸的条件。

### 3.1 解答

#### 步骤1：定义前向传播

$$
h_t = W_{hh}h_{t-1} + W_{hx}x_t, \quad o_t = W_{oh}h_t
$$

损失函数：
$$
L = \frac{1}{2}\sum_{t=1}^{T}(o_t - y_t)^2
$$

---

#### 步骤2：链式法则与 BPTT 展开

$W_{hh}$ 通过所有时间步的隐藏状态 $h_t$ 影响损失。令 $e_t = o_t - y_t$，则：

$$
\frac{\partial L}{\partial o_t} = o_t - y_t = e_t
$$

$$
\frac{\partial L}{\partial h_t} = \frac{\partial L}{\partial o_t} \frac{\partial o_t}{\partial h_t} + \frac{\partial L}{\partial h_{t+1}} \frac{\partial h_{t+1}}{\partial h_t}
$$

其中 $\frac{\partial o_t}{\partial h_t} = W_{oh}^T$，$\frac{\partial h_{t+1}}{\partial h_t} = W_{hh}^T$。

展开得到：
$$
\frac{\partial L}{\partial h_t} = \sum_{k=t}^{T} \left( \prod_{i=t+1}^{k} W_{hh}^T \right) W_{oh}^T e_k
$$

---

#### 步骤3：对 $W_{hh}$ 的梯度

$$
\frac{\partial h_k}{\partial W_{hh}} = h_{k-1}^T \otimes I \quad \text{（矩阵外积形式）}
$$

完整梯度为所有时间步的贡献之和：
$$
\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} \frac{\partial L}{\partial h_t} \cdot h_{t-1}^T
$$

代入 $\frac{\partial L}{\partial h_t}$ 得：
$$
\boxed{
\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} \sum_{k=t}^{T} \left(W_{hh}^T\right)^{k-t} W_{oh}^T e_k \cdot h_{t-1}^T
}
$$

---

#### 步骤4：梯度消失与梯度爆炸的条件

梯度表达式中出现了 $(W_{hh}^T)^{k-t}$（$W_{hh}$ 的幂次），其行为由 $W_{hh}$ 的特征值决定。设 $W_{hh}$ 的最大特征值为 $\lambda_{\max}$：

- **梯度消失**：当 $|\lambda_{\max}| < 1$ 时，$(W_{hh}^T)^{k-t}$ 随 $k-t$ 增大而趋于零矩阵，远期时间步对梯度的贡献指数级衰减，导致模型难以学习长期依赖。

- **梯度爆炸**：当 $|\lambda_{\max}| > 1$ 时，$(W_{hh}^T)^{k-t}$ 随 $k-t$ 增大而指数级增长，导致梯度值溢出，训练不稳定。

- **稳定传播**：仅当 $|\lambda_{\max}| \approx 1$ 时，梯度能在较长时间范围内稳定传播。

实践中，由于 $\tanh$ 等激活函数的饱和区导数接近零，梯度消失问题更为常见——这也是 LSTM/GRU 等门控结构被提出以缓解该问题的原因。

### 3.2 编程题

实现一个简单的 RNN 单元的前向传播和单步反向传播（仅计算梯度，不更新）。给定输入 $x_t$（形状 `(batch_size, input_size)`）、上一隐藏状态 $h_{prev}$（形状 `(batch_size, hidden_size)`），以及权重 $W_{hx}, W_{hh}, b_h$，计算当前隐藏状态 $h_t$。同时实现反向传播，已知上游梯度 `dh_next`（即损失对 $h_t$ 的梯度），计算 $dx_t, dh_{prev}, dW_{hx}, dW_{hh}, db_h$（使用 $\tanh$ 激活函数）。

In [2]:
import numpy as np

def rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h):
    """
    RNN 单元前向传播（tanh 激活）。
    参数:
        x_t: (batch_size, input_size)  当前时间步输入
        h_prev: (batch_size, hidden_size)  上一隐藏状态
        W_hx: (input_size, hidden_size)  输入到隐藏的权重
        W_hh: (hidden_size, hidden_size)  隐藏到隐藏的权重
        b_h: (hidden_size,)  隐藏层偏置
    返回:
        h_t: (batch_size, hidden_size)  当前隐藏状态
        cache: 用于反向传播的缓存
    """
    z = np.dot(x_t, W_hx) + np.dot(h_prev, W_hh) + b_h
    h_t = np.tanh(z)
    cache = (x_t, h_prev, W_hx, W_hh, b_h, z, h_t)
    return h_t, cache


def rnn_cell_backward(dh_next, cache):
    """
    RNN 单元单步反向传播。
    参数:
        dh_next: (batch_size, hidden_size)  损失对 h_t 的梯度（上游梯度）
        cache: 前向传播时保存的缓存
    返回:
        dx_t: (batch_size, input_size)  损失对输入 x_t 的梯度
        dh_prev: (batch_size, hidden_size)  损失对上一隐藏状态 h_prev 的梯度
        dW_hx: (input_size, hidden_size)  损失对权重 W_hx 的梯度
        dW_hh: (hidden_size, hidden_size)  损失对权重 W_hh 的梯度
        db_h: (hidden_size,)  损失对偏置 b_h 的梯度
    """
    x_t, h_prev, W_hx, W_hh, b_h, z, h_t = cache
    
    # tanh 的导数：d(tanh(z))/dz = 1 - tanh(z)^2 = 1 - h_t^2
    dz = dh_next * (1 - h_t ** 2)  # (batch_size, hidden_size)
    
    # 计算各梯度
    dx_t = np.dot(dz, W_hx.T)        # (batch_size, input_size)
    dh_prev = np.dot(dz, W_hh.T)      # (batch_size, hidden_size)
    dW_hx = np.dot(x_t.T, dz)         # (input_size, hidden_size)
    dW_hh = np.dot(h_prev.T, dz)      # (hidden_size, hidden_size)
    db_h = np.sum(dz, axis=0)         # (hidden_size,)
    
    return dx_t, dh_prev, dW_hx, dW_hh, db_h


# 测试代码
if __name__ == "__main__":
    # 设定维度
    batch_size, input_size, hidden_size = 3, 4, 5
    
    # 随机初始化输入和参数
    np.random.seed(42)
    x_t = np.random.randn(batch_size, input_size)
    h_prev = np.random.randn(batch_size, hidden_size)
    W_hx = np.random.randn(input_size, hidden_size)
    W_hh = np.random.randn(hidden_size, hidden_size)
    b_h = np.random.randn(hidden_size)
    
    # 前向传播
    h_t, cache = rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h)
    print("=== 前向传播 ===")
    print("h_t shape:", h_t.shape)  # 应为 (3, 5)
    print("h_t 值:\n", h_t)
    
    # 反向传播
    dh_next = np.random.randn(batch_size, hidden_size)
    dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_cell_backward(dh_next, cache)
    
    print("\n=== 反向传播 ===")
    print("dx_t shape:", dx_t.shape)       # 应为 (3, 4)
    print("dh_prev shape:", dh_prev.shape)  # 应为 (3, 5)
    print("dW_hx shape:", dW_hx.shape)      # 应为 (4, 5)
    print("dW_hh shape:", dW_hh.shape)      # 应为 (5, 5)
    print("db_h shape:", db_h.shape)        # 应为 (5,)
    print("db_h 值:", db_h)
    
    # 数值梯度检查（简单验证）
    print("\n=== 梯度数值检查 ===")
    epsilon = 1e-5
    # 检查 W_hx 的某个元素
    W_hx_plus = W_hx.copy()
    W_hx_plus[0, 0] += epsilon
    h_t_plus, _ = rnn_cell_forward(x_t, h_prev, W_hx_plus, W_hh, b_h)
    W_hx_minus = W_hx.copy()
    W_hx_minus[0, 0] -= epsilon
    h_t_minus, _ = rnn_cell_forward(x_t, h_prev, W_hx_minus, W_hh, b_h)
    numeric_grad = np.sum((h_t_plus - h_t_minus) * dh_next) / (2 * epsilon)
    print(f"W_hx[0,0] 数值梯度: {numeric_grad:.6f}")
    print(f"W_hx[0,0] 解析梯度: {dW_hx[0, 0]:.6f}")

=== 前向传播 ===
h_t shape: (3, 5)
h_t 值:
 [[ 0.37757047 -0.85065863 -0.99999996 -0.95887056  0.60632966]
 [-0.99893487 -0.99615252 -0.99992555  0.99878199 -0.02801693]
 [ 0.5842284   0.42268155 -0.9914534  -0.70369631 -0.77853071]]

=== 反向传播 ===
dx_t shape: (3, 4)
dh_prev shape: (3, 5)
dW_hx shape: (4, 5)
dW_hh shape: (5, 5)
db_h shape: (5,)
db_h 值: [-0.03669355 -0.41373297  0.00861537  0.03008595  1.52204834]

=== 梯度数值检查 ===
W_hx[0,0] 数值梯度: -0.229745
W_hx[0,0] 解析梯度: -0.229745


## 4. 高级循环神经网络

### 4.1 理论计算题

假设一个深度双向 RNN，有 $L$ 层，每层隐藏单元数为 $H$，输入维度为 $D$，输出维度为 $O$（仅考虑最后输出层）。计算该模型的参数总数（包括所有全连接层的权重和偏置），忽略嵌入层和输出层之前的投影，明确给出表达式。

### 4.1 解答

#### 第 1 层（$l=1$）

第 1 层接收原始输入 $x_t \in \mathbb{R}^{D}$，分别送入前向和后向 RNN：

**前向 RNN**：
- $W_{hh}^{(1,f)} \in \mathbb{R}^{H \times H}$：$H^2$ 个参数
- $W_{hx}^{(1,f)} \in \mathbb{R}^{H \times D}$：$H \cdot D$ 个参数
- $b_h^{(1,f)} \in \mathbb{R}^{H}$：$H$ 个参数

**后向 RNN**（与输入相同，独立参数）：
- $W_{hh}^{(1,b)} \in \mathbb{R}^{H \times H}$：$H^2$ 个参数
- $W_{hx}^{(1,b)} \in \mathbb{R}^{H \times D}$：$H \cdot D$ 个参数
- $b_h^{(1,b)} \in \mathbb{R}^{H}$：$H$ 个参数

第 1 层参数量：$2 \times (H^2 + H \cdot D + H) = 2H(H + D + 1)$

---

#### 第 $l$ 层（$l = 2, 3, \dots, L$）

第 $l$ 层的输入是第 $l-1$ 层前向和后向隐藏状态的拼接：
$$
h_{t}^{(l-1, \text{concat})} = [h_t^{(l-1,f)}; \, h_t^{(l-1,b)}] \in \mathbb{R}^{2H}
$$

**前向 RNN**（输入维度为 $2H$）：
- $W_{hh}^{(l,f)} \in \mathbb{R}^{H \times H}$：$H^2$
- $W_{hx}^{(l,f)} \in \mathbb{R}^{H \times 2H}$：$2H^2$
- $b_h^{(l,f)} \in \mathbb{R}^{H}$：$H$

**后向 RNN**（同理）：
- $W_{hh}^{(l,b)} \in \mathbb{R}^{H \times H}$：$H^2$
- $W_{hx}^{(l,b)} \in \mathbb{R}^{H \times 2H}$：$2H^2$
- $b_h^{(l,b)} \in \mathbb{R}^{H}$：$H$

每层（$l \ge 2$）参数量：$2 \times (H^2 + 2H^2 + H) = 2H(3H + 1)$

---

#### 输出层

最后输出层接收最后一层双向隐藏状态的拼接 $h_t^{(L,\text{concat})} \in \mathbb{R}^{2H}$，输出维度 $O$：
- 权重 $W_o \in \mathbb{R}^{O \times 2H}$：$2HO$ 个参数
- 偏置 $b_o \in \mathbb{R}^{O}$：$O$ 个参数

输出层参数量：$O(2H + 1)$

---

#### 参数总数

$$
\boxed{
\begin{aligned}
\text{Params}_{\text{total}} &= 2H(H + D + 1) && \text{（第1层）} \\
&+ 2(L-1) \cdot H(3H + 1) && \text{（第2至L层）} \\
&+ O(2H + 1) && \text{（输出层）}
\end{aligned}
}
$$

简化展开：
$$
\begin{aligned}
\text{Params}_{\text{total}} &= 2H^2 + 2HD + 2H + 6(L-1)H^2 + 2(L-1)H + 2HO + O \\
&= (6L - 4)H^2 + 2HD + 2LH + 2HO + O
\end{aligned}
$$

### 4.2 编程题

实现一个双向 RNN 编码器，接收序列 $X$（形状 `(seq_len, batch, input_dim)`），使用 `torch.nn.RNN` 或手动实现。要求返回每个时间步的拼接后的前向和后向隐藏状态（形状 `(seq_len, batch, 2*hidden_dim)`），以及最终时间步的拼接隐藏状态（作为序列表示）。

In [3]:
import torch
import torch.nn as nn


class BiRNNEncoder(nn.Module):
    """
    双向 RNN 编码器。
    返回每个时间步拼接后的前向+后向隐藏状态，
    以及最终时间步的拼接隐藏状态（作为序列表示）。
    """
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super(BiRNNEncoder, self).__init__()
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=False  # 输入形状为 (seq_len, batch, input_dim)
        )
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
    
    def forward(self, X):
        """
        参数:
            X: (seq_len, batch, input_dim)
        返回:
            outputs: (seq_len, batch, 2*hidden_dim) — 每个时间步的拼接隐藏状态
            final_hidden: (batch, 2*hidden_dim) — 最终时间步的拼接隐藏状态
        """
        # RNN 前向传播
        # outputs: (seq_len, batch, 2*hidden_dim)
        # h_n: (2*num_layers, batch, hidden_dim)
        outputs, h_n = self.rnn(X)
        
        # 最终时间步的拼接隐藏状态
        final_hidden = outputs[-1]  # (batch, 2*hidden_dim)
        
        return outputs, final_hidden


# 手动实现版本（不使用 torch.nn.RNN，便于理解）
class BiRNNEncoderManual(nn.Module):
    """手动实现双向 RNN 编码器"""
    def __init__(self, input_dim, hidden_dim):
        super(BiRNNEncoderManual, self).__init__()
        self.hidden_dim = hidden_dim
        
        # 前向 RNN 权重
        self.W_hx_f = nn.Linear(input_dim, hidden_dim, bias=True)
        self.W_hh_f = nn.Linear(hidden_dim, hidden_dim, bias=False)
        
        # 后向 RNN 权重
        self.W_hx_b = nn.Linear(input_dim, hidden_dim, bias=True)
        self.W_hh_b = nn.Linear(hidden_dim, hidden_dim, bias=False)
    
    def forward(self, X):
        seq_len, batch, _ = X.shape
        
        # 前向 RNN
        h_f = torch.zeros(batch, self.hidden_dim, device=X.device)
        forward_states = []
        for t in range(seq_len):
            h_f = torch.tanh(self.W_hx_f(X[t]) + self.W_hh_f(h_f))
            forward_states.append(h_f)
        forward_states = torch.stack(forward_states, dim=0)  # (seq_len, batch, hidden_dim)
        
        # 后向 RNN
        h_b = torch.zeros(batch, self.hidden_dim, device=X.device)
        backward_states = []
        for t in reversed(range(seq_len)):
            h_b = torch.tanh(self.W_hx_b(X[t]) + self.W_hh_b(h_b))
            backward_states.append(h_b)
        backward_states = torch.stack(backward_states[::-1], dim=0)  # 翻转回正向顺序
        
        # 拼接前向和后向
        outputs = torch.cat([forward_states, backward_states], dim=-1)  # (seq_len, batch, 2*hidden_dim)
        final_hidden = outputs[-1]  # (batch, 2*hidden_dim)
        
        return outputs, final_hidden


# 测试代码
if __name__ == "__main__":
    # 设置参数
    seq_len, batch, input_dim = 5, 3, 10
    hidden_dim = 8
    num_layers = 2
    
    torch.manual_seed(42)
    X = torch.randn(seq_len, batch, input_dim)
    
    # 使用 torch.nn.RNN
    print("=== 使用 torch.nn.RNN ===")
    encoder = BiRNNEncoder(input_dim, hidden_dim, num_layers=num_layers)
    outputs, final_hidden = encoder(X)
    print("输入 X shape:", X.shape)
    print("每个时间步输出 outputs shape:", outputs.shape)  # (5, 3, 16)
    print("最终隐藏状态 final_hidden shape:", final_hidden.shape)  # (3, 16)
    print("\n最终时间步拼接隐藏状态:\n", final_hidden)
    
    # 使用手动实现（单层）
    print("\n=== 手动实现（单层双向 RNN） ===")
    encoder_manual = BiRNNEncoderManual(input_dim, hidden_dim)
    outputs_manual, final_manual = encoder_manual(X)
    print("每个时间步输出 outputs shape:", outputs_manual.shape)  # (5, 3, 16)
    print("最终隐藏状态 final_hidden shape:", final_manual.shape)  # (3, 16)
    print("\n最终时间步拼接隐藏状态:\n", final_manual)

=== 使用 torch.nn.RNN ===
输入 X shape: torch.Size([5, 3, 10])
每个时间步输出 outputs shape: torch.Size([5, 3, 16])
最终隐藏状态 final_hidden shape: torch.Size([3, 16])

最终时间步拼接隐藏状态:
 tensor([[-0.7315,  0.4562, -0.7423,  0.2948,  0.2706,  0.8011, -0.0629,  0.0421,
          0.2064,  0.4449, -0.3537, -0.1589,  0.6120, -0.2793,  0.8483, -0.0737],
        [-0.5397,  0.0027,  0.5783,  0.3968, -0.1491, -0.1330, -0.0460,  0.7442,
          0.4149, -0.1629, -0.2711,  0.0576, -0.6778, -0.2009,  0.1484, -0.3937],
        [-0.3838, -0.7054, -0.1732,  0.0296,  0.5991,  0.6779, -0.1741,  0.5224,
         -0.1886, -0.1141, -0.0534,  0.3549,  0.0751, -0.3495,  0.3523, -0.2809]],
       grad_fn=<SelectBackward0>)

=== 手动实现（单层双向 RNN） ===
每个时间步输出 outputs shape: torch.Size([5, 3, 16])
最终隐藏状态 final_hidden shape: torch.Size([3, 16])

最终时间步拼接隐藏状态:
 tensor([[ 0.0327, -0.1300,  0.3294, -0.8053, -0.3693, -0.5581,  0.5886,  0.4981,
          0.6313,  0.6475, -0.8262, -0.0183, -0.0048, -0.3041,  0.0997, -0.1454],
        [ 0.56

## 5. 嵌入向量

### 5.1 理论计算题

在 Skip-gram 模型中，给定中心词 $w_c$ 和上下文词 $w_o$，使用负采样（采样 $K$ 个负样本）。推导其损失函数（对数似然）的表达式，并说明如何从噪声分布中采样负样本。假设词向量为 $v_c, u_o$，负样本词向量为 $u_{n_k}$，写出完整的目标函数。

### 5.1 解答

#### 1. Skip-gram 模型的概率定义

给定中心词 $w_c$ 和上下文词 $w_o$，模型希望最大化正样本 $ (w_c, w_o) $ 一起出现的概率：
$$
P(+|w_c, w_o) = \sigma(v_c \cdot u_o) = \frac{1}{1 + \exp(-v_c \cdot u_o)}
$$

其中 $\sigma$ 为 sigmoid 函数，$v_c$ 是中心词的嵌入向量，$u_o$ 是上下文词的嵌入向量。

---

#### 2. 负采样

对于每个正样本 $(w_c, w_o)$，从噪声分布 $P_n(w)$ 中采样 $K$ 个负样本词 $w_{n_1}, w_{n_2}, \dots, w_{n_K}$。负样本的概率为：
$$
P(-|w_c, w_{n_k}) = \sigma(-v_c \cdot u_{n_k}) = 1 - \sigma(v_c \cdot u_{n_k})
$$

---

#### 3. 目标函数（最大化对数似然）

对于单个正样本 $(w_c, w_o)$ 和 $K$ 个负样本，对数似然为：
$$
\mathcal{L} = \log \sigma(v_c \cdot u_o) + \sum_{k=1}^{K} \mathbb{E}_{w_{n_k} \sim P_n(w)} \left[ \log \sigma(-v_c \cdot u_{n_k}) \right]
$$

对应的**损失函数**（最小化负对数似然）为：
$$
\boxed{
J = -\log \sigma(v_c \cdot u_o) - \sum_{k=1}^{K} \log \sigma(-v_c \cdot u_{n_k})
}
$$

---

#### 4. 噪声分布采样

Mikolov 等人（2013）提出使用**一元词频的 $3/4$ 次幂**作为噪声分布：
$$
P_n(w) = \frac{U(w)^{3/4}}{\sum_{w'} U(w')^{3/4}}
$$

其中 $U(w)$ 是词 $w$ 在语料库中的一元词频。

使用 $3/4$ 次幂的原因：
- 相比于均匀分布，保留了词频的相对差异，高频词更可能被选为负样本（符合直觉）
- 相比于原始词频分布，$3/4$ 次幂使低频词的采样概率相对提升，避免过度偏向最高频的几个词
- 经验表明 $3/4$ 次幂在大多数数据集上效果最好

---

#### 5. 完整目标函数（整个语料库）

对整个语料库中所有中心词 $w_c$ 及所有上下文窗口内的 $w_o$：
$$
J_{\text{total}} = -\sum_{(w_c, w_o) \in D} \left[ \log \sigma(v_c \cdot u_o) + \sum_{k=1}^{K} \log \sigma(-v_c \cdot u_{n_k}) \right]
$$

其中 $D$ 是所有正样本对（中心词，上下文词）的集合，每个正样本对随机采样 $K$ 个独立的负样本。

### 5.2 编程题

实现 CBOW 模型的前向传播和损失计算（不使用负采样，仅用完整 softmax）。给定一批上下文词的索引列表（每个样本有 `context_size` 个上下文词），词汇表大小 $V$，嵌入维度 $d$。输入权重矩阵 $W$（形状 $(V, d)$）和输出权重矩阵 $W_{out}$（形状 $(d, V)$）。计算平均上下文向量作为隐藏层，然后计算输出概率分布，并计算交叉熵损失（目标为中心词索引）。返回损失值。

In [4]:
import torch
import torch.nn.functional as F


def cbow_forward_loss(context_indices, target_indices, W, W_out):
    """
    CBOW 模型前向传播和交叉熵损失计算（完整 softmax）。
    参数:
        context_indices: (batch_size, context_size) — 上下文词索引
        target_indices: (batch_size,) — 中心词索引（目标）
        W: (V, d) — 输入嵌入矩阵
        W_out: (d, V) — 输出权重矩阵
    返回:
        loss: 标量，交叉熵损失值
    """
    # 1. 查表获取上下文词的嵌入向量
    # context_indices: (batch_size, context_size)
    # W[context_indices]: (batch_size, context_size, d)
    context_emb = W[context_indices]
    
    # 2. 计算平均上下文向量作为隐藏层表示
    h = context_emb.mean(dim=1)  # (batch_size, d)
    
    # 3. 计算输出 logits
    logits = torch.matmul(h, W_out)  # (batch_size, V)
    
    # 4. 计算 softmax 概率分布
    probs = F.softmax(logits, dim=1)  # (batch_size, V)
    
    # 5. 计算交叉熵损失
    # 交叉熵 = -log(prob[target_class])
    log_probs = torch.log(probs + 1e-10)  # 防止 log(0)
    loss = -log_probs[torch.arange(len(target_indices)), target_indices].mean()
    
    return loss


# 测试代码
if __name__ == "__main__":
    # 设定超参数
    V = 1000      # 词汇表大小
    d = 50        # 嵌入维度
    context_size = 4  # 每个样本的上下文词数
    batch_size = 8
    
    torch.manual_seed(42)
    
    # 随机初始化参数
    W = torch.randn(V, d)           # 输入嵌入矩阵
    W_out = torch.randn(d, V)       # 输出权重矩阵
    
    # 随机生成一批上下文词索引和对应的中心词索引
    context_indices = torch.randint(0, V, (batch_size, context_size))
    target_indices = torch.randint(0, V, (batch_size,))
    
    # 前向传播和损失计算
    loss = cbow_forward_loss(context_indices, target_indices, W, W_out)
    
    print("=== CBOW 前向传播 ===")
    print("上下文词索引形状:", context_indices.shape)
    print("目标中心词索引形状:", target_indices.shape)
    print("嵌入矩阵 W 形状:", W.shape)
    print("输出权重矩阵 W_out 形状:", W_out.shape)
    print(f"\n交叉熵损失值: {loss.item():.6f}")
    
    # 额外验证：改变 batch_size 测试
    print("\n=== 不同 batch_size 测试 ===")
    for bs in [1, 4, 16]:
        ctx_idx = torch.randint(0, V, (bs, context_size))
        tgt_idx = torch.randint(0, V, (bs,))
        loss_bs = cbow_forward_loss(ctx_idx, tgt_idx, W, W_out)
        print(f"batch_size={bs}, loss={loss_bs.item():.6f}")

=== CBOW 前向传播 ===
上下文词索引形状: torch.Size([8, 4])
目标中心词索引形状: torch.Size([8])
嵌入矩阵 W 形状: torch.Size([1000, 50])
输出权重矩阵 W_out 形状: torch.Size([50, 1000])

交叉熵损失值: 10.308250

=== 不同 batch_size 测试 ===
batch_size=1, loss=14.779563
batch_size=4, loss=12.409395
batch_size=16, loss=11.717522


## 6. 注意力机制

### 6.1 理论计算题

给定查询矩阵 $Q \in \mathbb{R}^{2 \times 4}$，键矩阵 $K \in \mathbb{R}^{3 \times 4}$，值矩阵 $V \in \mathbb{R}^{3 \times 5}$。计算缩放点积注意力（无掩码）的输出矩阵，要求写出中间步骤（先计算得分矩阵，再 softmax，再加权求和）。使用 $\text{score} = QK^T / \sqrt{d_k}$（$d_k = 4$）。可以只列出数值计算过程（用符号或具体数值）。

### 6.1 解答

为便于数值计算，构造以下具体矩阵：

$$
Q = \begin{bmatrix} 1 & 0 & 1 & 0 \\ 0 & 1 & 0 & 1 \end{bmatrix} \in \mathbb{R}^{2 \times 4}
$$

$$
K = \begin{bmatrix} 1 & 0 & 1 & 0 \\ 0 & 1 & 0 & 1 \\ 1 & 1 & 0 & 0 \end{bmatrix} \in \mathbb{R}^{3 \times 4}
$$

$$
V = \begin{bmatrix} 1 & 0 & 0 & 0 & 1 \\ 0 & 1 & 0 & 1 & 0 \\ 0 & 0 & 1 & 0 & 1 \end{bmatrix} \in \mathbb{R}^{3 \times 5}
$$

---

#### 步骤1：计算得分矩阵 $S = QK^T / \sqrt{d_k}$

缩放因子：$\sqrt{d_k} = \sqrt{4} = 2$

$$
QK^T = \begin{bmatrix} 1 & 0 & 1 & 0 \\ 0 & 1 & 0 & 1 \end{bmatrix}
\begin{bmatrix} 1 & 0 & 1 \\ 0 & 1 & 1 \\ 1 & 0 & 0 \\ 0 & 1 & 0 \end{bmatrix}
= \begin{bmatrix} 2 & 0 & 1 \\ 0 & 2 & 1 \end{bmatrix}
$$

$$
S = \frac{QK^T}{2} = \begin{bmatrix} 1.0 & 0.0 & 0.5 \\ 0.0 & 1.0 & 0.5 \end{bmatrix}
$$

---

#### 步骤2：Softmax 归一化（按行）

对得分矩阵 $S$ 的每一行应用 softmax：

**第 1 行** $[1.0, 0.0, 0.5]$：
$$
\begin{aligned}
e^{1.0} &= 2.718, \quad e^{0.0} = 1.000, \quad e^{0.5} = 1.649 \\
\sum &= 2.718 + 1.000 + 1.649 = 5.367 \\
\text{softmax} &= [0.506, \; 0.186, \; 0.307]
\end{aligned}
$$

**第 2 行** $[0.0, 1.0, 0.5]$：
$$
\begin{aligned}
e^{0.0} &= 1.000, \quad e^{1.0} = 2.718, \quad e^{0.5} = 1.649 \\
\sum &= 1.000 + 2.718 + 1.649 = 5.367 \\
\text{softmax} &= [0.186, \; 0.506, \; 0.307]
\end{aligned}
$$

注意力权重矩阵：
$$
A = \text{softmax}(S) = \begin{bmatrix} 0.506 & 0.186 & 0.307 \\ 0.186 & 0.506 & 0.307 \end{bmatrix}
$$

---

#### 步骤3：加权求和 $\text{Output} = A \cdot V$

$$
\begin{aligned}
\text{Output}[0,:] &= 0.506 \cdot [1,0,0,0,1] + 0.186 \cdot [0,1,0,1,0] + 0.307 \cdot [0,0,1,0,1] \\
&= [0.506, \; 0.186, \; 0.307, \; 0.186, \; 0.813]
\end{aligned}
$$

$$
\begin{aligned}
\text{Output}[1,:] &= 0.186 \cdot [1,0,0,0,1] + 0.506 \cdot [0,1,0,1,0] + 0.307 \cdot [0,0,1,0,1] \\
&= [0.186, \; 0.506, \; 0.307, \; 0.506, \; 0.493]
\end{aligned}
$$

---

#### 最终输出

$$
\boxed{
\text{Output} = \begin{bmatrix} 0.506 & 0.186 & 0.307 & 0.186 & 0.813 \\ 0.186 & 0.506 & 0.307 & 0.506 & 0.493 \end{bmatrix} \in \mathbb{R}^{2 \times 5}
}
$$

### 6.2 编程题

实现多头注意力（Multi-Head Attention）的前向传播，假设 `num_heads=2`，`d_model=4`。给定输入 $X$（形状 `(seq_len, batch, d_model)`），分别线性投影得到 $Q, K, V$（每个头的维度 $d_k = d_v = d_{model} / num\_heads$）。对每个头计算缩放点积注意力，然后将所有头的输出拼接并经过最终线性层。返回输出（形状与输入相同）。

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


def multi_head_attention_forward(X, num_heads=2, d_model=4):
    """
    多头注意力前向传播。
    参数:
        X: (seq_len, batch, d_model) — 输入序列
        num_heads: 头数，默认 2
        d_model: 模型维度，默认 4
    返回:
        output: (seq_len, batch, d_model) — 多头注意力输出
    """
    seq_len, batch, _ = X.shape
    d_k = d_model // num_heads  # 每个头的维度 = 2
    d_v = d_model // num_heads  # 每个头的值维度 = 2
    
    # 1. 线性投影得到 Q, K, V（使用可学习权重）
    # 在实际实现中，通常将多个头的投影合并
    W_Q = nn.Linear(d_model, d_model, bias=False)
    W_K = nn.Linear(d_model, d_model, bias=False)
    W_V = nn.Linear(d_model, d_model, bias=False)
    W_O = nn.Linear(d_model, d_model, bias=False)
    
    # 投影
    Q = W_Q(X)  # (seq_len, batch, d_model)
    K = W_K(X)  # (seq_len, batch, d_model)
    V = W_V(X)  # (seq_len, batch, d_model)
    
    # 2. 重塑为多头形式：(seq_len, batch, num_heads, d_k)
    # 然后转置为 (batch, num_heads, seq_len, d_k) 以便批量计算注意力
    Q = Q.view(seq_len, batch, num_heads, d_k).permute(1, 2, 0, 3)  # (batch, num_heads, seq_len, d_k)
    K = K.view(seq_len, batch, num_heads, d_k).permute(1, 2, 0, 3)  # (batch, num_heads, seq_len, d_k)
    V = V.view(seq_len, batch, num_heads, d_v).permute(1, 2, 0, 3)  # (batch, num_heads, seq_len, d_v)
    
    # 3. 对每个头计算缩放点积注意力
    scale = math.sqrt(d_k)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / scale       # (batch, num_heads, seq_len, seq_len)
    attn_weights = F.softmax(scores, dim=-1)                     # (batch, num_heads, seq_len, seq_len)
    attn_output = torch.matmul(attn_weights, V)                  # (batch, num_heads, seq_len, d_v)
    
    # 4. 拼接所有头的输出
    attn_output = attn_output.permute(2, 0, 1, 3).contiguous()  # (seq_len, batch, num_heads, d_v)
    attn_output = attn_output.view(seq_len, batch, d_model)      # (seq_len, batch, d_model)
    
    # 5. 最终线性投影
    output = W_O(attn_output)  # (seq_len, batch, d_model)
    
    return output


# 测试代码
if __name__ == "__main__":
    # 设定参数
    seq_len, batch_size = 3, 2
    d_model = 4
    num_heads = 2
    
    torch.manual_seed(42)
    X = torch.randn(seq_len, batch_size, d_model)
    
    print("=== 多头注意力前向传播 ===")
    print("输入 X shape:", X.shape)
    print("d_model:", d_model, "| num_heads:", num_heads)
    print("每个头的维度 d_k = d_v =", d_model // num_heads)
    
    output = multi_head_attention_forward(X, num_heads, d_model)
    print("\n输出 shape:", output.shape)  # 应为 (3, 2, 4)
    print("\n输入 X:\n", X)
    print("\n输出:\n", output)
    
    # 验证输出形状与输入相同
    assert output.shape == X.shape, f"输出形状 {output.shape} 与输入形状 {X.shape} 不一致!"
    print("\n✓ 输出形状与输入形状一致")

=== 多头注意力前向传播 ===
输入 X shape: torch.Size([3, 2, 4])
d_model: 4 | num_heads: 2
每个头的维度 d_k = d_v = 2

输出 shape: torch.Size([3, 2, 4])

输入 X:
 tensor([[[ 1.9269,  1.4873,  0.9007, -2.1055],
         [ 0.6784, -1.2345, -0.0431, -1.6047]],

        [[ 0.3559, -0.6866, -0.4934,  0.2415],
         [-1.1109,  0.0915, -2.3169, -0.2168]],

        [[-0.3097, -0.3957,  0.8034, -0.6216],
         [-0.5920, -0.0631, -0.8286,  0.3309]]])

输出:
 tensor([[[-0.2397, -0.2013, -0.1152, -0.4099],
         [-0.2113,  0.0121,  0.2835, -0.2767]],

        [[-0.2822, -0.2615, -0.1421, -0.4736],
         [-0.1520, -0.0290,  0.1019, -0.2596]],

        [[-0.2301, -0.2152, -0.0427, -0.3770],
         [-0.1889,  0.0789,  0.3545, -0.2062]]], grad_fn=<UnsafeViewBackward0>)

✓ 输出形状与输入形状一致
